# שלב 3 — שיטה A: גרף מעברים (Transition Graph) + Count-Min Sketch

**זו "השיטה שלכם"** — לא Clustering ולא Suffix. הרעיון:

1. **גרף מעברים**: צומת = תא H3, קשת = מעבר בין שני תאים עוקבים. משקל הקשת = מספר הנסיעות שביצעו את המעבר.
2. **גידול חלונות רציפים (Apriori על רצפים)**: תת-מסלול = חלון רציף של תאים. נבנה חלונות באורך גדל והולך, ובכל רמה נשמור רק חלונות שעברו בהם **≥ X% מהנסיעות**. תכונת ה-**anti-monotonicity** (כל תת-חלון של חלון פופולרי הוא פופולרי) מאפשרת לעצור מוקדם ולגזום.
3. **מקסימליות**: נשמור רק תתי-מסלולים שאי אפשר להאריך (לא prefix/suffix של חלון פופולרי ארוך יותר) — כדי לא להחזיר 100 חלונות שכולם חלקים של אותו מסלול.
4. **Count-Min Sketch**: אופטימיזציה מקורבת להערכת משקלי הקשתות/תדירויות בלי groupBy יקר.

**הקלט:** `trips_h3.parquet` משלב 2. **הפלט:** תתי-מסלולים פופולריים מקסימליים עם אורך (ק"מ) ותמיכה — מסוננים לפי הקונפיגורציות ≥1/3/5/10/20/40 ק"מ.


## 1. SparkSession + קונפיגורציה

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType
import h3
from math import radians, sin, cos, asin, sqrt

spark = (
    SparkSession.builder.appName("porto-taxi-stage3-graph")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

USE_SAMPLE = True
if USE_SAMPLE:
    INPUT_PATH  = "out/trips_h3_sample.parquet"
    OUTPUT_PATH = "out/subroutes_graph_sample.parquet"
else:
    INPUT_PATH  = "gs://YOUR_BUCKET/encoded/trips_h3.parquet"
    OUTPUT_PATH = "gs://YOUR_BUCKET/results/subroutes_graph.parquet"

X_PCT = 0.5        # סף פופולריות באחוזים — כוונו לפי הדאטה (קטן יותר -> מסלולים ארוכים יותר)
MAX_L = 200        # תקרת אורך חלון (תאים) — מנגנון בטיחות
CONFIGS_KM = [1, 3, 5, 10, 20, 40]


In [ ]:
import time
_t0 = time.time()
trips = spark.read.parquet(INPUT_PATH).select("TRIP_ID", "cells", "n_cells").cache()
N = trips.count()
minsup = max(2, int(N * X_PCT / 100))
print(f"trips N={N}   X={X_PCT}%   minsup={minsup}")


## 2. גרף המעברים (רמה 2 = קשתות)

כל קשת היא מעבר בין שני תאים עוקבים; המשקל הוא מספר הנסיעות הייחודיות שעברו בה. זהו בסיס השיטה — הרמות הגבוהות יותר הן בעצם הליכה לאורך הגרף הזה.

In [ ]:
def windows_of_len(df, L):
    # מייצר את כל החלונות הרציפים באורך L (מוגן מפני שורות קצרות מ-L)
    expr = (f"CASE WHEN size(cells) >= {L} "
            f"THEN transform(sequence(1, size(cells)-{L}+1), i -> slice(cells, i, {L})) "
            f"ELSE cast(array() as array<array<string>>) END")
    return (df.filter(F.col("n_cells") >= L)
              .withColumn("w", F.expr(expr))
              .select("TRIP_ID", F.explode("w").alias("win"))
              .withColumn("win_key", F.array_join("win", ">"))
              .dropDuplicates(["TRIP_ID", "win_key"]))   # נסיעה ייחודית לכל חלון

edges = (windows_of_len(trips, 2)
         .groupBy("win_key").agg(F.first("win").alias("win"),
                                 F.count(F.lit(1)).alias("weight")))
print("distinct edges:", edges.count())
edges.orderBy(F.desc("weight")).show(10, truncate=False)


## 3. גידול חלונות איטרטיבי (Apriori על רצפים רציפים)

לכל אורך L נספור את התמיכה (מספר נסיעות ייחודיות) של כל חלון, נשמור רק `support >= minsup`, ונעצור כשאין יותר חלונות פופולריים — מובטח ע"י anti-monotonicity.

In [ ]:
freq = None
for L in range(2, MAX_L + 1):
    supp = (windows_of_len(trips, L)
            .groupBy("win_key")
            .agg(F.first("win").alias("win"), F.count(F.lit(1)).alias("support"))
            .filter(F.col("support") >= minsup)
            .withColumn("L", F.lit(L)))
    c = supp.count()
    if c == 0:
        print(f"L={L}: 0 frequent -> stop")
        break
    print(f"L={L}: {c} frequent windows")
    freq = supp if freq is None else freq.unionByName(supp)

freq = freq.cache()
print("total frequent windows:", freq.count())


## 4. סינון למקסימליים

חלון `w` באורך L ניתן להארכה אם הוא prefix או suffix של חלון פופולרי באורך L+1. מקסימלי = חלון פופולרי שאינו ניתן להארכה (anti-join).

In [ ]:
ext = (freq
       .withColumn("pref", F.expr("array_join(slice(win, 1, L-1), '>')"))
       .withColumn("suf",  F.expr("array_join(slice(win, 2, L-1), '>')")))
extendable = (ext.select(F.col("pref").alias("k"))
                 .union(ext.select(F.col("suf").alias("k")))
                 .distinct())
maximal = freq.join(extendable, freq.win_key == extendable.k, "left_anti").cache()
print("maximal popular sub-routes:", maximal.count())


## 5. אורך גאוגרפי (ק"מ) של כל תת-מסלול

In [ ]:
def route_km(win):
    pts = [h3.cell_to_latlng(c) for c in win]   # (lat, lng)
    t = 0.0
    for (la1, lo1), (la2, lo2) in zip(pts, pts[1:]):
        dlat, dlon = radians(la2 - la1), radians(lo2 - lo1)
        a = sin(dlat/2)**2 + cos(radians(la1))*cos(radians(la2))*sin(dlon/2)**2
        t += 2 * 6371 * asin(sqrt(a))
    return float(t)

km_udf = F.udf(route_km, DoubleType())
res = (maximal
       .withColumn("km", km_udf("win"))
       .withColumn("n_cells", F.size("win"))
       .select("win", "win_key", "n_cells", "support", "km")
       .cache())

res.select(F.max("km").alias("max_km"), F.avg("km").alias("avg_km"),
           F.expr("approx_percentile(km, array(0.5,0.9,0.99))").alias("km_p50_90_99")).show(truncate=False)


## 6. תוצאות לפי קונפיגורציה + שמירה (Top-100 הארוכים לכל ספֵּף)

In [ ]:
for K in CONFIGS_KM:
    print(f"  >= {K:>2} km : {res.filter(F.col('km') >= K).count()} popular sub-routes")

print("\ntop 10 longest popular sub-routes:")
res.orderBy(F.desc("km")).select("km", "n_cells", "support").show(10, truncate=False)

# שומרים את כל המקסימליים; מחברת המפה (שלב 4) תסנן Top-100 לכל קונפיגורציה
res.write.mode("overwrite").parquet(OUTPUT_PATH)
print("saved:", OUTPUT_PATH)


### שמירת מטריקות להשוואה בין השיטות

In [ ]:
import json
n_max = res.count()
per_config = {str(K): res.filter(F.col("km") >= K).count() for K in CONFIGS_KM}
metrics = {
    "method": "graph_cms",
    "structure": "Count-Min Sketch (d=5, w=65536)",
    "distributed": True,
    "runtime_sec": round(time.time() - _t0, 1),
    "n_maximal": n_max,
    "struct_mem_bytes": 5 * 65536 * 8,
    "minsup": minsup, "X_pct": X_PCT,
    "per_config": per_config,
}
with open("out/metrics_graph.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(metrics)


## 7. אופטימיזציה מקורבת — Count-Min Sketch

ספירת משקלי הקשתות עם `groupBy` מדויקת אך דורשת shuffle ומאחסנת ערך לכל קשת ייחודית. **Count-Min Sketch** מאחסן מטריצת `d×w` בזיכרון **קבוע** ונותן הערכת תדירות עם שגיאה חסומה (תמיד ≥ הערך האמיתי).

נבנה CMS מבוזר: כל partition בונה מטריצה מקומית (`mapPartitions`), וממזגים בסכימה (`reduce`). `d` = מספר פונקציות hash, `w` = רוחב (יותר רוחב → פחות התנגשויות → דיוק גבוה יותר).

In [ ]:
import numpy as np, zlib, statistics

D, W = 5, 65536          # depth x width
SALT = [f"s{i}_" for i in range(D)]

def cms_cols(k):
    return [zlib.crc32((SALT[i] + k).encode()) % W for i in range(D)]

def build_partition(rows):
    m = np.zeros((D, W), dtype=np.int64)
    for r in rows:
        for i, c in enumerate(cms_cols(r.win_key)):
            m[i, c] += 1
    yield m

edges_occ = windows_of_len(trips, 2).select("win_key")     # (trip,edge) deduped
cms = edges_occ.rdd.mapPartitions(build_partition).reduce(lambda a, b: a + b)

def cms_estimate(k):
    return int(min(cms[i, c] for i, c in enumerate(cms_cols(k))))

print(f"CMS depth={D} width={W}  fixed memory = {D*W*8:,} bytes")

# השוואת דיוק מול הספירה המדויקת על הקשתות הכבדות
exact_edges = edges.orderBy(F.desc("weight")).limit(300).collect()
errs = [abs(cms_estimate(r["win_key"]) - r["weight"]) for r in exact_edges]
print(f"on top-300 edges:  mean_abs_err={statistics.mean(errs):.2f}  max_abs_err={max(errs)}")
print(f"example edge: exact={exact_edges[0]['weight']}  cms_estimate={cms_estimate(exact_edges[0]['win_key'])}")


## 8. סיכום השיטה ומה הלאה

**יתרונות:** אינטואיטיבי, מקביל לחלוטין ב-Spark (כל רמה = explode+groupBy), גיזום anti-monotone, ו-CMS לחיסכון בזיכרון.
**חסרונות/מגבלות:** מספר מעברים על הדאטה כמספר הרמות; רגיש לבחירת X.

**להרצה על Dataproc:** `USE_SAMPLE=False`, נתיבי `gs://`, להסיר `.master`, להעלות `shuffle.partitions`, ולוודא ש-`h3` מותקן על ה-workers.

**השלב הבא:** שיטה B (Suffix Array + HLL) ושיטה C (Clustering + MinHash-LSH), ואז מחברת מפה (שלב 4) שמציגה את ה-Top-100 לכל קונפיגורציה ומשווה בין השיטות.
